# Camp QMIND mini demo: tune a feature up and down

Individual, self-contained demo for [MVP.md](MVP.md) on **Qwen3-1.7B-Base** with a Qwen-Scope sparse
autoencoder (SAE). In four steps we load the model, find a feature for a concept, see what it fires on, and
clamp it up and down while the model writes. How the clamp works is explained in [steering.md](steering.md).

Needs a GPU with roughly 4 GB free (CPU works but is slow). Nothing here has been run on the real model yet.
The probe sentences are drafts nobody has read, so read them before trusting any result. Per
[AGENTS.md](AGENTS.md), do not quote numbers from this notebook unless you ran it.

In [1]:
# !pip install torch "transformers>=4.51" huggingface_hub
import random
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-1.7B-Base"
SAE_REPO = "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50"  # check the exact name in the Qwen-Scope collection
LAYER = 14   # a middle layer; a guess, try a few
K = 50       # TopK sparsity of this SAE (the L0_50 in the name)
CONCEPT = "queens"  # "queens" or "bridge"

## 1. Load the model and the SAE

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to(device).eval()

weights = torch.load(hf_hub_download(SAE_REPO, f"layer{LAYER}.sae.pt"), map_location=device)
sae = {name: t.float() for name, t in weights.items()}  # W_enc (d_sae, d_model), W_dec (d_model, d_sae)
print({name: tuple(t.shape) for name, t in sae.items()})

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

{'W_enc': (32768, 2048), 'W_dec': (2048, 32768), 'b_enc': (32768,), 'b_dec': (2048,)}


Helpers. `encode` turns a hidden vector into sparse feature activations, `decode` turns them back,
`residual` grabs the hidden state after a layer, and `clamp` is the steering hook.

In [3]:
def encode(x):
    pre = x.float() @ sae["W_enc"].T + sae["b_enc"]
    vals, idx = pre.topk(K, dim=-1)
    return torch.zeros_like(pre).scatter_(-1, idx, vals)

def decode(acts):
    return acts @ sae["W_dec"].T + sae["b_dec"]

def residual(text, layer=LAYER):
    ids = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = model(**ids, output_hidden_states=True).hidden_states
    return hidden[layer + 1][0], tok.convert_ids_to_tokens(ids["input_ids"][0])  # hidden[0] is embeddings

def generate(prompt, n=60):
    ids = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)

def clamp(feature, value, layer=LAYER):
    """Set one feature to `value` at every position. Returns a handle; call .remove() to undo."""
    def hook(_module, _inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        act = encode(h)[..., feature]
        h2 = h + ((value - act).unsqueeze(-1) * sae["W_dec"][:, feature]).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2
    return model.model.layers[layer].register_forward_hook(hook)

## Sanity check: which residual stream is "layer N"?

The Qwen-Scope model card does not say whether `layerN` means the input or the output of block N. The right
choice should reconstruct the hidden state far better. **If both errors are high, stop here.**

In [4]:
text = "The Golden Gate Bridge opened in 1937 and spans the strait at San Francisco."
for name, layer in (("output of block", LAYER), ("input to block", LAYER - 1)):
    x, _ = residual(text, layer)
    err = ((x.float() - decode(encode(x))).norm() / x.float().norm()).item()
    print(f"reconstruction error, {name} {LAYER}: {err:.3f}")

reconstruction error, output of block 14: 0.820
reconstruction error, input to block 14: 0.820


## 2. Find a feature for a concept

Each concept has **target** sentences (should fire), **near-miss** sentences (close in topic, should not
fire) and shared **neutral** sentences. We score every feature by its mean peak activation on target minus
neutral, then show the near-miss value next to it. A good candidate is high on target and low on both others.

In [5]:
NEUTRAL = [
    "The recipe calls for two cups of flour and a pinch of salt.",
    "The train to the airport leaves every twenty minutes.",
    "She repaired the bicycle chain with a small wrench.",
    "The stock market closed slightly higher on Tuesday.",
    "He practiced the piano for an hour before dinner.",
    "The new phone has a larger battery and a faster camera.",
    "They painted the fence white before the neighbours arrived.",
    "The committee will meet again on the first Monday of March.",
]
PROBES = {
    "queens": {
        "target": [
            "Queen's University in Kingston, Ontario was founded in 1841.",
            "She spent four years at Queen's studying engineering and living near campus.",
            "Frosh Week at Queen's starts with students in tricolour jackets.",
            "The Queen's Gaels won the football game at Richardson Stadium.",
            "Grant Hall is the clock tower building at the heart of the Queen's campus.",
            "He applied to Queen's Commerce and hoped to study at Goodes Hall.",
            "Homecoming at Queen's brings thousands of alumni back to Kingston.",
            "Stauffer Library stays packed during exams at Queen's.",
        ],
        "near_miss": [
            "Queen Elizabeth II was crowned in 1953 at Westminster Abbey.",
            "The band Queen played Wembley Stadium in 1986.",
            "The queen bee lays up to two thousand eggs a day.",
            "In chess, the queen can move any number of squares in a straight line.",
            "The University of Toronto is the largest university in Ontario.",
            "Waterloo is known for its co-op engineering programs.",
            "Kingston has a famous limestone waterfront and a historic penitentiary.",
            "McGill University is in Montreal and is known for its research.",
        ],
    },
    "bridge": {
        "target": [
            "The Golden Gate Bridge opened in 1937 and spans the strait at San Francisco Bay.",
            "We drove across the Golden Gate Bridge as fog rolled in from the Pacific.",
            "The Golden Gate Bridge is painted a colour called International Orange.",
            "Thousands of tourists walk the Golden Gate Bridge every day.",
            "From Marin County you can see the Golden Gate Bridge towers rising out of the mist.",
            "Joseph Strauss was the chief engineer of the Golden Gate Bridge.",
            "The Golden Gate Bridge was the longest suspension bridge in the world when it opened.",
            "Cyclists cross the Golden Gate Bridge to reach Sausalito.",
        ],
        "near_miss": [
            "The Brooklyn Bridge connects Manhattan and Brooklyn across the East River.",
            "The Bay Bridge links San Francisco to Oakland.",
            "Alcatraz Island sits in the middle of San Francisco Bay.",
            "Fog is common along the California coast in summer.",
            "The Tower Bridge in London opens to let ships pass.",
            "Cable cars climb the steep hills of San Francisco.",
            "The Sydney Harbour Bridge is nicknamed the Coathanger.",
            "Lombard Street in San Francisco is famous for its sharp curves.",
        ],
    },
}

sets = {**PROBES[CONCEPT], "neutral": NEUTRAL}
peak = {name: torch.stack([encode(residual(s)[0]).max(0).values for s in sents]) for name, sents in sets.items()}
mean = {name: p.mean(0) for name, p in peak.items()}
score = mean["target"] - mean["neutral"]

print(f"{'feature':>8} {'score':>8} {'target':>8} {'near':>8} {'neutral':>8} {'max seen':>9}")
best = score.topk(10).indices.tolist()
for f in best:
    top = max(p[:, f].max().item() for p in peak.values())
    print(f"{f:>8} {score[f]:>8.2f} {mean['target'][f]:>8.2f} {mean['near_miss'][f]:>8.2f} "
          f"{mean['neutral'][f]:>8.2f} {top:>9.2f}")

 feature    score   target     near  neutral  max seen
   28345   155.75   311.44   145.48   155.69   1245.98
   31205   154.11   154.11   304.02     0.00   1232.86
   32494   147.77   147.77     0.00     0.00   1182.19
    8520    70.08  5664.28  5517.91  5594.20   5996.63
   32044    51.04    51.04     7.57     0.00     65.30
    7662    48.33  3757.15  3660.63  3708.82   3965.13
   21977    41.84    41.84    17.91     0.00     46.09
   23819    40.94    40.94     0.00     0.00     44.03
   22548    36.62  2991.52  2914.29  2954.90   3168.08
   13920    31.34  2657.90  2591.33  2626.56   2809.05


Pick a feature from the table. By default we take the top-scoring one, but look at the near-miss column
first: a feature that fires just as hard on the near-miss sentences is probably the wrong one.

In [6]:
FEATURE = best[0]  # change this to another index from the table if the top one looks wrong
MAX_ACT = max(p[:, FEATURE].max().item() for p in peak.values())
print(f"feature {FEATURE}, max activation seen {MAX_ACT:.2f}")

feature 28345, max activation seen 1245.98


## 3. What does it actually fire on?

For each sentence, the token where the feature peaks. Do this before believing any label for the feature.

In [7]:
for name in ("target", "near_miss"):
    print(f"--- {name} ---")
    for s in sets[name]:
        x, tokens = residual(s)
        acts = encode(x)[:, FEATURE]
        i = acts.argmax().item()
        print(f"{acts[i].item():6.1f}  on {tokens[i]!r:<14} {s}")

--- target ---
   0.0  on 'Queen'        Queen's University in Kingston, Ontario was founded in 1841.
   0.0  on 'She'          She spent four years at Queen's studying engineering and living near campus.
   0.0  on 'F'            Frosh Week at Queen's starts with students in tricolour jackets.
   0.0  on 'The'          The Queen's Gaels won the football game at Richardson Stadium.
   0.0  on 'Grant'        Grant Hall is the clock tower building at the heart of the Queen's campus.
1245.5  on 'He'           He applied to Queen's Commerce and hoped to study at Goodes Hall.
   0.0  on 'Home'         Homecoming at Queen's brings thousands of alumni back to Kingston.
1246.0  on 'St'           Stauffer Library stays packed during exams at Queen's.
--- near_miss ---
   0.0  on 'Queen'        Queen Elizabeth II was crowned in 1953 at Westminster Abbey.
   0.0  on 'The'          The band Queen played Wembley Stadium in 1986.
   0.0  on 'The'          The queen bee lays up to two thousand eggs a

## 4. Steer it

We generate from the same prompt with no steering, then with the feature clamped at several fractions of its
max (0 removes it), then with a **random feature** clamped just as hard as a control. If the random feature
does the same thing, the effect is just "we pushed the hidden state somewhere strange". Greedy decoding, one
prompt: read every output, it is a demo and not evidence.

In [8]:
PROMPT = "Q: What should I study in university?\nA:"
MULTIPLES = [0, 0.5, 1, 2, 5]

print("--- no steering ---")
print(generate(PROMPT), "\n")
for m in MULTIPLES:
    handle = clamp(FEATURE, m * MAX_ACT)
    out = generate(PROMPT)
    handle.remove()
    print(f"--- feature {FEATURE} at {m}x max ---\n{out}\n")

control = random.Random(0).randrange(sae["W_enc"].shape[0])
handle = clamp(control, max(MULTIPLES) * MAX_ACT)
out = generate(PROMPT)
handle.remove()
print(f"--- CONTROL: random feature {control} at {max(MULTIPLES)}x ---\n{out}")

--- no steering ---
 I don't know what you want to do, but if you want to be a doctor, you should study medicine. If you want to be a lawyer, you should study law. If you want to be a teacher, you should study education. If you want to be a scientist, you should 

--- feature 28345 at 0x max ---
 I don't know what you want to do, but if you want to be a doctor, you should study medicine. If you want to be a lawyer, you should study law. If you want to be a teacher, you should study education. If you want to be a scientist, you should

--- feature 28345 at 0.5x max ---
 the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

--- feature 28345 at 1x max ---
 the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the th

### To "tune it down"

Use a prompt where the feature should matter, such as `"Q: Tell me about Queen's University in Kingston.\nA:"`,
and compare no steering with `m = 0`. If the answer gets vaguer or drifts to a generic university, the
feature is doing causal work. Also try the other concept (`CONCEPT = "bridge"`) and a prompt-only baseline
("Talk about the Golden Gate Bridge in every answer.") to see whether plain prompting beats steering, as
AxBench reports.

### Caveats

- The SAE repo name, file names and layer are unverified guesses. The reconstruction check above is the
  safety net.
- Probe sets are tiny and unreviewed. A feature that tops 8 sentences can still be a mislabel, which is the
  point of the audit in MVP.md.
- Setting a feature to 0 removes one direction only. A weak "down" effect does not mean the feature is
  unimportant (see steering.md).